# 서론 수정 전 점검 3종 (TQP v0.4 결과 기준)

| 절 | 확인 내용 | 필요한 것 | GPU |
|---|---|---|---|
| 1 | counterbalanced B2의 WHILE 실패가 **해석 오류**인지 **구현 오류**인지 | v0.4 예측 CSV(`llm_predictions_v04.csv`) | 불필요 |
| 2 | 탐침을 **해석 탐침 / 구현 탐침**으로 나눈 v0.5 채점으로 개발 세트를 다시 채점 (재생성 없음) | 같은 CSV | 불필요 |
| 3 | RefAV 공식 프롬프트 API에 **시간 순서 함수**가 있는지, RefAV val 설명에 시간 접속사가 얼마나 있는지 | GitHub 접근, (선택) HF 접근 | 불필요 |

- 모델을 다시 돌리지 않습니다. v0.4에서 저장한 원출력(`raw_output`)만 다시 채점합니다.
- 2절 결과는 **개발 세트(288문장) 재채점**이므로 탐색적 수치입니다. 본실험 채점 규칙을 정하는 근거로만 쓰십시오.
- 런타임 유형은 CPU로 충분합니다. 위에서부터 순서대로 실행하십시오.

## 0. 설정

In [1]:
from pathlib import Path
import sys, os, json, re, ast, subprocess, importlib

IN_COLAB = "google.colab" in sys.modules
DRIVE_ROOT = Path("/content/drive/MyDrive")
if IN_COLAB and not DRIVE_ROOT.is_dir():
    from google.colab import drive
    drive.mount("/content/drive")

WORK = Path("/content") if Path("/content").is_dir() else Path.cwd()

# ---- 필요하면 직접 지정 (None이면 자동 탐색) ----
TQP_NOTEBOOK = None   # 예: DRIVE_ROOT / "Colab Notebooks/TQP_TEG_AV2_v03_Colab.ipynb"
PRED_CSV = None       # 예: DRIVE_ROOT / "AV2_TQP_TEG_Results/v04_val_20261016_045848/llm_predictions_v04.csv"
RESULTS_ROOT = DRIVE_ROOT / "AV2_TQP_TEG_Results"
REFAV_REPO_URL = "https://github.com/CainanD/RefAV.git"
HF_REPO = "CainanD/RefAV"

OUT_DIR = (RESULTS_ROOT if RESULTS_ROOT.is_dir() else WORK) / "v05_checks"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("작업 폴더:", WORK, "| 결과 폴더:", OUT_DIR)

Mounted at /content/drive
작업 폴더: /content | 결과 폴더: /content/drive/MyDrive/AV2_TQP_TEG_Results/v05_checks


In [2]:
# tqp_teg.py / tqp_nl.py 준비: 작업 폴더에 없으면 v0.3 노트북의 %%writefile 셀에서 추출
MODULES = ("tqp_teg.py", "tqp_nl.py")


def extract_modules(nb_path: Path, dest: Path) -> None:
    nb = json.loads(Path(nb_path).read_text(encoding="utf-8"))
    found = {}
    for c in nb["cells"]:
        src = "".join(c["source"]) if isinstance(c["source"], list) else c["source"]
        m = re.match(r"%%writefile\s+(tqp_teg\.py|tqp_nl\.py)\s*\n", src)
        if m:
            found[m.group(1)] = src[m.end():]
    missing = [m for m in MODULES if m not in found]
    assert not missing, f"{nb_path} 에서 {missing} 셀을 찾지 못했습니다."
    for name, body in found.items():
        (dest / name).write_text(body, encoding="utf-8")
    print("추출 완료:", nb_path)


if not all((WORK / m).exists() for m in MODULES):
    nb_path = Path(TQP_NOTEBOOK) if TQP_NOTEBOOK else None
    if nb_path is None:
        cands = sorted((DRIVE_ROOT / "Colab Notebooks").glob("TQP_TEG_AV2_v03*.ipynb")) if DRIVE_ROOT.is_dir() else []
        cands += sorted(WORK.glob("TQP_TEG_AV2_v03*.ipynb"))
        assert cands, ("tqp_teg.py / tqp_nl.py 가 없습니다. TQP_NOTEBOOK 에 v0.3 노트북 경로를 지정하거나, "
                       "노트북 파일을 작업 폴더에 올려 주십시오.")
        nb_path = cands[-1]
    extract_modules(nb_path, WORK)
else:
    print("기존 모듈 사용:", [str(WORK / m) for m in MODULES])

if str(WORK) not in sys.path:
    sys.path.insert(0, str(WORK))
import tqp_teg as T
import tqp_nl as N
importlib.reload(T); importlib.reload(N)
print("엔진 버전:", getattr(T, "VERSION", "?"))

추출 완료: /content/drive/MyDrive/Colab Notebooks/TQP_TEG_AV2_v03_Colab.ipynb
엔진 버전: 0.3.0


In [3]:
# v0.4 예측 CSV 불러오기 + 저장된 점수 재현 확인
import numpy as np
import pandas as pd


def find_pred_csv() -> Path:
    if PRED_CSV:
        return Path(PRED_CSV)
    cands = []
    for root in (WORK / "tqp_teg_out", RESULTS_ROOT):
        if root.is_dir():
            cands += list(root.glob("v04_*/llm_predictions_v04.csv"))
    assert cands, "llm_predictions_v04.csv 를 찾지 못했습니다. PRED_CSV 에 경로를 지정하십시오."
    return max(cands, key=lambda p: p.stat().st_mtime)


csv_path = find_pred_csv()
res = pd.read_csv(csv_path)
res["raw_output"] = res["raw_output"].fillna("").astype(str)
for col in ("exact", "probe_exact"):
    res[col] = res[col].fillna(False).astype(bool)
P_NAME, B2_NAME = "P (TQP)", "B2 (Python)"
VARIANT_ORDER = [v for v in ("v1_original", "zero_shot", "balanced", "counterbalanced") if v in set(res.variant)]
print("CSV:", csv_path)
print(res.groupby(["variant", "method"]).size().unstack())


def gold_of(r):
    return T.Query(r.gold_first, r.gold_relation, r.gold_second)


# 저장된 exact / probe_exact 가 현재 모듈로 그대로 재현되는지 확인 (채점 코드가 바뀌지 않았는지 점검)
_probe_cache = {}
re_exact, re_probe = [], []
for r in res.itertuples():
    g = gold_of(r)
    probes = _probe_cache.setdefault(g, N.build_probes(g))
    if r.method == P_NAME:
        pred, _, _ = N.parse_prediction(r.raw_output)
        sc = N.score_tqp(g, pred)
        ps = N.probe_score(g, N.tqp_probe_accepts(pred, probes), probes)
    else:
        acc, _ = N.run_b2(r.raw_output, probes)
        sc = N.score_b2(g, acc)
        ps = N.probe_score(g, acc, probes)
    re_exact.append(sc["exact"]); re_probe.append(ps["probe_exact"])
res["re_exact"], res["re_probe_exact"] = re_exact, re_probe
mis = res[(res.re_exact != res.exact) | (res.re_probe_exact != res.probe_exact)]
print(f"재현 불일치: {len(mis)} / {len(res)}")
if len(mis):
    print("주의: 저장 당시와 채점 결과가 다릅니다. 모듈 버전을 확인하십시오.")
    display(mis.groupby(["variant", "method"]).size())

CSV: /content/drive/MyDrive/AV2_TQP_TEG_Results/v04_val_20261016_045848/llm_predictions_v04.csv
method           B2 (Python)  P (TQP)
variant                              
balanced                 288      288
counterbalanced          288      288
zero_shot                288      288
재현 불일치: 0 / 1728


## 1. B2 WHILE 실패의 원인

정답이 WHILE인 B2 출력을 아래 4개 탐침에 다시 실행합니다.

| 탐침 | 구간 (X = 정답 첫 사건, Y = 둘째 사건) | 의미 |
|---|---|---|
| `while_same` | X 2–6초, Y 2–6초 | 두 구간이 동일. 겹침 비율의 분모를 무엇으로 잡아도 1.0 |
| `while_cur` | X 2–6초, Y 2.5–5.5초 | v0.4의 while 탐침. 짧은 구간 기준 1.0, 긴 구간·IoU 기준 0.75 |
| `then_xy`, `then_yx` | 간격 2초, 겹침 없음 | THEN으로 해석했는지 확인 |

분류 규칙

- **OK**: `while_same`, `while_cur` 모두 수락, THEN 탐침 거부
- **IMPL (구현 오류)**: `while_same` 수락, `while_cur` 거부, THEN 탐침 거부 → WHILE로 해석했지만 겹침 비율을 다르게 구현
- **INTERP (해석 오류)**: THEN 탐침 수락, `while_same` 거부 → THEN으로 해석(예시의 THEN 코드를 복사했을 가능성)
- **NONE**: 아무것도 수락하지 않음 → 사건 유형 문자열 오류 또는 판정 논리 파손. 아래 코드 표본으로 확인
- **FAIL**: 코드 추출·AST 검사·실행 실패

In [4]:
def mk_events(name, specs):
    evs = [T.Event(f"{name}:{i}", "probe", name, t, float(s), float(e)) for i, (t, s, e) in enumerate(specs)]
    dicts = [{"type": e.type, "start": e.start, "end": e.end} for e in evs]
    return evs, dicts


def while_diag_probes(g):
    x, y = g.first, g.second
    specs = {"while_same": [(x, 2, 6), (y, 2, 6)],
             "while_cur": [(x, 2, 6), (y, 2.5, 5.5)],
             "then_xy": [(x, 1, 3), (y, 5, 7)],
             "then_yx": [(y, 1, 3), (x, 5, 7)]}
    return [(n, *mk_events(n, s), None) for n, s in specs.items()]


def classify_while(acc):
    if acc is None:
        return "FAIL"
    then_any = acc["then_xy"] or acc["then_yx"]
    if acc["while_same"] and acc["while_cur"] and not then_any:
        return "OK"
    if acc["while_same"] and not acc["while_cur"] and not then_any:
        return "IMPL"
    if then_any and not acc["while_same"]:
        return "INTERP"
    if not any(acc.values()):
        return "NONE"
    return "MIXED"


def static_features(text):
    try:
        c = N.extract_code(text)
    except Exception:
        return {"code": "", "has_0.8": False, "has_5s": False, "types_in_code": ""}
    return {"code": c, "has_0.8": "0.8" in c, "has_5s": bool(re.search(r"(?<![\d.])5(\.0)?(?![\d.])", c)),
            "types_in_code": ",".join(t for t in T.EVENT_TYPES if t in c)}


wb = res[(res.method == B2_NAME) & (res.gold_relation == "WHILE")].copy()
rows = []
for r in wb.itertuples():
    g = gold_of(r)
    acc, err = N.run_b2(r.raw_output, while_diag_probes(g))
    rows.append({"while_class": classify_while(acc), "diag_error": err, **static_features(r.raw_output),
                 **({f"acc_{k}": v for k, v in acc.items()} if acc else {})})
wb = pd.concat([wb.reset_index(drop=True), pd.DataFrame(rows)], axis=1)

print("[1-a] 조건별 분류 (B2, 정답 WHILE 문장)")
display(pd.crosstab([wb.variant], wb.while_class, margins=True).reindex(VARIANT_ORDER + ["All"]))
print("\n[1-b] 조건 × 사건 쌍 × 분류")
wb["pair"] = wb.gold_first + "+" + wb.gold_second
display(pd.crosstab([wb.variant, wb.pair], wb.while_class))
print("\n[1-c] 정적 특징: 코드에 0.8(WHILE 조건) / 5초(THEN 조건)가 있는 비율")
display(wb.groupby(["variant", "while_class"])[["has_0.8", "has_5s"]].mean().round(2)
        .assign(n=wb.groupby(["variant", "while_class"]).size()))

[1-a] 조건별 분류 (B2, 정답 WHILE 문장)


while_class,FAIL,INTERP,MIXED,NONE,OK,All
variant,,,,,,
zero_shot,10,16,10,9,3,48
balanced,0,16,0,0,32,48
counterbalanced,0,48,0,0,0,48
All,10,80,10,9,35,144



[1-b] 조건 × 사건 쌍 × 분류


while_class                      FAIL  INTERP  MIXED  NONE  OK
variant         pair                                          
balanced        ACCELERATE+TURN     0       0      0     0  12
                DECELERATE+TURN     0       8      0     0   4
                TURN+ACCELERATE     0       0      0     0  12
                TURN+DECELERATE     0       8      0     0   4
counterbalanced ACCELERATE+TURN     0      12      0     0   0
                DECELERATE+TURN     0      12      0     0   0
                TURN+ACCELERATE     0      12      0     0   0
                TURN+DECELERATE     0      12      0     0   0
zero_shot       ACCELERATE+TURN     4       2      3     3   0
                DECELERATE+TURN     4       2      2     4   0
                TURN+ACCELERATE     2       7      2     0   1
                TURN+DECELERATE     0       5      3     2   2


[1-c] 정적 특징: 코드에 0.8(WHILE 조건) / 5초(THEN 조건)가 있는 비율


has_0.8  has_5s   n
variant         while_class                     
balanced        INTERP          0.00    1.00  16
                OK              1.00    0.00  32
counterbalanced INTERP          0.00    1.00  48
zero_shot       FAIL            0.80    0.20  10
                INTERP          0.19    0.25  16
                MIXED           0.00    0.70  10
                NONE            0.00    0.33   9
                OK              0.00    0.00   3

In [5]:
# [1-d] counterbalanced 분류별 코드 표본 (분류당 최대 3개). 직접 읽고 분류가 맞는지 확인하십시오.
SHOW_VARIANT = "counterbalanced"
for cls, g in wb[wb.variant == SHOW_VARIANT].groupby("while_class"):
    print("=" * 100)
    print(f"[{SHOW_VARIANT}] {cls}: {len(g)}개")
    for r in g.head(3).itertuples():
        print("-" * 100)
        print("문장:", r.sentence, "| 정답:", r.gold_first, "WHILE", r.gold_second, "| 오류:", r.diag_error)
        print(r.code if r.code else r.raw_output[:800])
wb.drop(columns=["code"]).to_csv(OUT_DIR / "check1_b2_while_diagnosis.csv", index=False)
print("\n저장:", OUT_DIR / "check1_b2_while_diagnosis.csv")

[counterbalanced] INTERP: 48개
----------------------------------------------------------------------------------------------------
문장: A vehicle slows down while turning. | 정답: DECELERATE WHILE TURN | 오류: None
def matches(events):
    for a in events:
        for b in events:
            if a is b or a['type'] != 'DECELERATE' or b['type'] != 'TURN':
                continue
            overlap = max(0.0, min(a['end'], b['end']) - max(a['start'], b['start']))
            if a['start'] < b['start'] and overlap <= 0.5 and -0.5 <= b['start'] - a['end'] <= 5.0:
                return True
    return False

----------------------------------------------------------------------------------------------------
문장: A vehicle slows down while taking a turn. | 정답: DECELERATE WHILE TURN | 오류: None
def matches(events):
    for a in events:
        for b in events:
            if a is b or a['type'] != 'DECELERATE' or b['type'] != 'TURN':
                continue
            overlap = max(0.0, min(a['

### 1절 결과 읽는 법

- **INTERP가 대부분**: B2가 WHILE 문장을 THEN으로 해석했습니다. counterbalanced 예시에서 THEN 예시 코드가 더 많아 복사가 일어났을 가능성이 큽니다. 서론과 H2에는 "예시의 관계 구성을 따라가는 경향"으로 쓰십시오.
- **IMPL이 대부분**: B2는 WHILE로 해석했지만 겹침 비율을 다르게 구현했고, v0.4 채점은 이를 관계 오류로 집계했습니다. 이 경우 B2의 v0.4 관계·해석 정확도는 과소평가되어 있으므로 2절의 v0.5 채점이 필요합니다.
- **NONE이 대부분**: 코드 표본을 직접 읽어야 합니다. 사건 유형 문자열을 틀렸다면 해석 오류(사건), 논리가 깨졌다면 구현 오류입니다.

## 2. v0.5 탐침으로 개발 세트 재채점 (재생성 없음)

탐침을 두 묶음으로 나눕니다.

**해석 탐침**: 어떤 합리적인 구현에서도 답이 같도록 경계에서 멀리 둔 탐침
- `then_xy`, `then_yx`: 간격 2초, 겹침 없음
- `while_same`: 동일 구간
- `x_other`, `other_y`: 한쪽 사건을 무관 유형으로 바꾼 것 (사건 유형 하나만 검사하는 코드를 잡음)

**구현 탐침**: 판정 규칙의 경계를 확인하는 탐침
- `far` (간격 8초), `then_gap45` (간격 4.5초), `then_ovl03` (0.3초 겹침, THEN 허용 범위)
- `while_unequal` (v0.4 while 탐침), `partial` (겹침 2초, 비율 0.33: 어느 관계도 아님)
- `only_x`, `only_y`, `other`

**지표**
- 해석 정확도 `interp_exact`: 해석 탐침 3종으로 추론한 관계·방향이 정답과 같고, 사건 유형 탐침 2종을 거부
- 판정 정확도 `judge_exact`: 모든 탐침에서 실행기와 같은 판정
- 구현 오류 `impl_err` = 해석은 맞고 판정은 틀림

각 탐침의 정답은 실행기(`query_accepts`)로 계산하고, 설계 의도와 일치하는지 먼저 검사합니다.

In [6]:
INTERP_PROBES = ("then_xy", "then_yx", "while_same", "x_other", "other_y")
IMPL_PROBES = ("far", "then_gap45", "then_ovl03", "while_unequal", "partial", "only_x", "only_y", "other")
ALL_PROBES = INTERP_PROBES + IMPL_PROBES


def build_probes_v05(g):
    x, y = g.first, g.second
    o = next(t for t in T.EVENT_TYPES if t not in (x, y))
    tx, ty = ((1, 3), (5, 7)) if g.relation == "THEN" else ((2, 6), (2, 6))
    specs = {
        "then_xy": [(x, 1, 3), (y, 5, 7)],
        "then_yx": [(y, 1, 3), (x, 5, 7)],
        "while_same": [(x, 2, 6), (y, 2, 6)],
        "x_other": [(x, *tx), (o, *ty)],
        "other_y": [(o, *tx), (y, *ty)],
        "far": [(x, 1, 2), (y, 10, 12)],
        "then_gap45": [(x, 1, 2), (y, 6.5, 8)],
        "then_ovl03": [(x, 1, 3), (y, 2.7, 5)],
        "while_unequal": [(x, 2, 6), (y, 2.5, 5.5)],
        "partial": [(x, 0, 6), (y, 4, 10)],
        "only_x": [(x, 1, 3)],
        "only_y": [(y, 5, 7)],
        "other": [(o, 1, 3), (o, 5, 7)],
    }
    out = []
    for n in ALL_PROBES:
        evs, dicts = mk_events(n, specs[n])
        out.append((n, evs, dicts, N.query_accepts(g, evs)))
    return out


# 설계 의도 검사: 실행기 판정이 아래 기대값과 다르면 중단
EXPECT = {
    "THEN": dict(then_xy=True, then_yx=False, while_same=False, x_other=False, other_y=False, far=False,
                 then_gap45=True, then_ovl03=True, while_unequal=False, partial=False,
                 only_x=False, only_y=False, other=False),
    "WHILE": dict(then_xy=False, then_yx=False, while_same=True, x_other=False, other_y=False, far=False,
                  then_gap45=False, then_ovl03=False, while_unequal=True, partial=False,
                  only_x=False, only_y=False, other=False),
}
for g in {gold_of(r) for r in res.itertuples()}:
    got = {n: acc for n, _, _, acc in build_probes_v05(g)}
    bad = {n: (got[n], EXPECT[g.relation][n]) for n in ALL_PROBES if got[n] != EXPECT[g.relation][n]}
    assert not bad, f"{g}: 실행기 판정이 설계와 다릅니다 (실제, 기대) {bad}"
print("탐침 설계 검사 통과:", len({gold_of(r) for r in res.itertuples()}), "개 정답 질의")

탐침 설계 검사 통과: 16 개 정답 질의


In [7]:
def infer_v05(acc):
    key = (acc["then_xy"], acc["then_yx"], acc["while_same"])
    return {(True, False, False): "THEN_XY", (False, True, False): "THEN_YX",
            (False, False, True): "WHILE"}.get(key, "OTHER")


def score_v05(g, acc, probes):
    if acc is None:
        return {"v05_valid": False, "v05_inferred": None, "interp_exact": False, "judge_exact": False,
                "impl_fail": None}
    inferred = infer_v05(acc)
    want = "WHILE" if g.relation == "WHILE" else "THEN_XY"
    types_ok = not (acc["x_other"] or acc["other_y"])
    interp = inferred == want and types_ok
    judge = all(acc[n] == gold for n, _, _, gold in probes)
    failed = [n for n, _, _, gold in probes if n in IMPL_PROBES and acc[n] != gold]
    return {"v05_valid": True, "v05_inferred": inferred, "interp_exact": interp, "judge_exact": judge,
            "impl_fail": ";".join(failed)}


_cache5 = {}
scores = []
for r in res.itertuples():
    g = gold_of(r)
    probes = _cache5.setdefault(g, build_probes_v05(g))
    if r.method == P_NAME:
        pred, _, _ = N.parse_prediction(r.raw_output)
        acc = {n: N.query_accepts(pred, evs) for n, evs, _, _ in probes} if pred else None
    else:
        acc, _ = N.run_b2(r.raw_output, probes)
    scores.append(score_v05(g, acc, probes))
res5 = pd.concat([res.reset_index(drop=True), pd.DataFrame(scores)], axis=1)
res5["impl_err"] = res5.interp_exact & ~res5.judge_exact

# 점검: P는 실행기가 판정하므로 interp_exact == judge_exact == v0.4 exact 여야 함
p = res5[res5.method == P_NAME]
print("P 점검 | interp≠judge:", int((p.interp_exact != p.judge_exact).sum()),
      "| interp≠v0.4 exact:", int((p.interp_exact != p.exact).sum()))

P 점검 | interp≠judge: 0 | interp≠v0.4 exact: 0


In [8]:
def summarize5(df):
    def agg(g):
        return pd.Series({
            "n": len(g),
            "v04_exact": g.exact.mean(), "v04_probe_exact": g.probe_exact.mean(),
            "v05_interp": g.interp_exact.mean(), "v05_judge": g.judge_exact.mean(),
            "impl_err": g.impl_err.mean(),
        }).round(3)
    return df.groupby(["variant", "method"]).apply(agg).reindex(VARIANT_ORDER, level=0)


print("[2-a] v0.4 대 v0.5 채점 (개발 세트, 탐색적)")
display(summarize5(res5))

print("\n[2-b] 정답 관계별 (v0.5 해석 정확도)")
display(res5.pivot_table(index=["variant", "method"], columns="gold_relation", values="interp_exact",
                         aggfunc="mean").round(3).reindex(VARIANT_ORDER, level=0))

print("\n[2-c] P 대 B2 정확 McNemar: 해석(interp)과 판정(judge) 각각")
mc = []
for v in VARIANT_ORDER:
    sub = res5[res5.variant == v]
    a = sub[sub.method == P_NAME].set_index("sentence")
    b = sub[sub.method == B2_NAME].set_index("sentence").reindex(a.index)
    if len(a) == 0 or b.interp_exact.isna().any():
        continue
    for metric in ("interp_exact", "judge_exact"):
        mc.append({"variant": v, "metric": metric, "P": round(a[metric].mean(), 3),
                   "B2": round(b[metric].astype(bool).mean(), 3),
                   **N.mcnemar_exact(a[metric].tolist(), b[metric].astype(bool).tolist())})
display(pd.DataFrame(mc))

[2-a] v0.4 대 v0.5 채점 (개발 세트, 탐색적)


/tmp/ipykernel_613/1873761741.py:9: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return df.groupby(["variant", "method"]).apply(agg).reindex(VARIANT_ORDER, level=0)


n  v04_exact  v04_probe_exact  v05_interp  \
variant         method                                                       
zero_shot       B2 (Python)  288.0      0.069            0.017       0.073   
                P (TQP)      288.0      0.177            0.177       0.177   
balanced        B2 (Python)  288.0      0.854            0.854       0.854   
                P (TQP)      288.0      0.913            0.913       0.913   
counterbalanced B2 (Python)  288.0      0.726            0.726       0.726   
                P (TQP)      288.0      0.681            0.681       0.681   

                             v05_judge  impl_err  
variant         method                            
zero_shot       B2 (Python)      0.003     0.069  
                P (TQP)          0.177     0.000  
balanced        B2 (Python)      0.854     0.000  
                P (TQP)          0.913     0.000  
counterbalanced B2 (Python)      0.726     0.000  
                P (TQP)          0.681     0.000


[2-b] 정답 관계별 (v0.5 해석 정확도)


gold_relation                 THEN  WHILE
variant         method                   
zero_shot       B2 (Python)  0.075  0.062
                P (TQP)      0.158  0.271
balanced        B2 (Python)  0.892  0.667
                P (TQP)      0.904  0.958
counterbalanced B2 (Python)  0.871  0.000
                P (TQP)      0.617  1.000


[2-c] P 대 B2 정확 McNemar: 해석(interp)과 판정(judge) 각각


,variant,metric,P,B2,P_only_correct,B2_only_correct,p_value
0,zero_shot,interp_exact,0.177,0.073,47,17,0.0002
1,zero_shot,judge_exact,0.177,0.003,51,1,0.0000
2,balanced,interp_exact,0.913,0.854,33,16,0.0213
3,balanced,judge_exact,0.913,0.854,33,16,0.0213
4,counterbalanced,interp_exact,0.681,0.726,57,70,0.2869
5,counterbalanced,judge_exact,0.681,0.726,57,70,0.2869


In [9]:
b2 = res5[res5.method == B2_NAME]

print("[2-d] B2 구현 오류: 해석은 맞고 판정이 틀린 문장에서 틀린 구현 탐침")
fails = (b2[b2.impl_err].assign(probe=lambda d: d.impl_fail.str.split(";")).explode("probe"))
if len(fails):
    display(pd.crosstab(fails.variant, fails.probe).reindex(VARIANT_ORDER).fillna(0).astype(int))
else:
    print("없음")

print("\n[2-e] v0.4 → v0.5 해석 판정이 바뀐 B2 문장 수 (행: v0.4 exact, 열: v0.5 interp)")
for v in VARIANT_ORDER:
    s = b2[b2.variant == v]
    print(f"--- {v}")
    display(pd.crosstab(s.exact.rename("v0.4 exact"), s.interp_exact.rename("v0.5 interp")))

print("\n[2-f] 바뀐 사례 표본 (최대 8개)")
changed = b2[b2.exact != b2.interp_exact]
display(changed[["variant", "sentence", "gold_first", "gold_relation", "gold_second",
                 "pred", "v05_inferred", "impl_fail"]].head(8))

res5.drop(columns=["re_exact", "re_probe_exact"]).to_csv(OUT_DIR / "check2_rescored_v05_dev.csv", index=False)
(OUT_DIR / "check2_probe_spec_v05.json").write_text(json.dumps(
    {"interp_probes": INTERP_PROBES, "impl_probes": IMPL_PROBES, "expect": EXPECT}, indent=2), encoding="utf-8")
print("저장:", OUT_DIR / "check2_rescored_v05_dev.csv")

[2-d] B2 구현 오류: 해석은 맞고 판정이 틀린 문장에서 틀린 구현 탐침


probe,far,partial,then_gap45,then_ovl03,while_unequal
variant,,,,,
zero_shot,15,5,1,5,4
balanced,0,0,0,0,0
counterbalanced,0,0,0,0,0



[2-e] v0.4 → v0.5 해석 판정이 바뀐 B2 문장 수 (행: v0.4 exact, 열: v0.5 interp)
--- zero_shot


v0.5 interp,False,True
v0.4 exact,,
False,264,4
True,3,17


--- balanced


v0.5 interp,False,True
v0.4 exact,,
False,42,0
True,0,246


--- counterbalanced


v0.5 interp,False,True
v0.4 exact,,
False,79,0
True,0,209



[2-f] 바뀐 사례 표본 (최대 8개)


,variant,sentence,gold_first,gold_relation,gold_second,pred,v05_inferred,impl_fail
323,zero_shot,A vehicle accelerates before coming to a stop.,ACCELERATE,THEN,STOP,OTHER,THEN_XY,far;while_unequal;partial
345,zero_shot,A vehicle turns after coming to a stop.,STOP,THEN,TURN,STOP THEN TURN,OTHER,far
429,zero_shot,"Before stopping, a vehicle accelerates.",ACCELERATE,THEN,STOP,OTHER,THEN_XY,far;while_unequal;partial
436,zero_shot,"After turning, a vehicle speeds up.",TURN,THEN,ACCELERATE,TURN THEN ACCELERATE,THEN_XY,far
438,zero_shot,"After taking a turn, a vehicle speeds up.",TURN,THEN,ACCELERATE,TURN THEN ACCELERATE,THEN_XY,far
512,zero_shot,A vehicle speeds up and then stops.,ACCELERATE,THEN,STOP,OTHER,THEN_XY,far;while_unequal;partial
516,zero_shot,A vehicle stops and then turns.,STOP,THEN,TURN,OTHER,THEN_XY,while_unequal;partial


저장: /content/drive/MyDrive/AV2_TQP_TEG_Results/v05_checks/check2_rescored_v05_dev.csv


### 2절 결과 읽는 법

- **B2의 `v05_interp`가 `v04_exact`보다 크게 높음**: v0.4 채점이 구현 오류를 해석 오류로 집계하고 있었습니다. v0.5로 바꾸고, 서론 5문단의 "해석 정확도와 판정 정확도를 분리한다"를 그대로 쓸 수 있습니다.
- **두 값이 거의 같음**: v0.4의 해석 추론이 이미 구현 세부에 흔들리지 않았다는 뜻입니다. v0.5로 바꾸는 이득은 `x_other`, `other_y`로 사건 유형 검사가 엄격해지는 것 정도입니다. 이 경우에도 해석/판정 분리 보고는 유지할 가치가 있습니다.
- **`impl_err`가 zero-shot에서만 큼**: 예시 코드가 B2의 구현 부담을 없앤다는 해석을 뒷받침합니다. H2의 메커니즘 서술에 쓸 수 있는 관찰입니다(개발 세트이므로 정성적으로만).
- P는 실행기가 판정하므로 `impl_err`가 항상 0이어야 합니다. 0이 아니면 채점 코드를 점검하십시오.
- v0.5를 채택한다면 `build_probes_v05`, `score_v05`를 `tqp_nl.py`에 옮겨 버전을 올리고, 본실험 전에 고정하십시오.

## 3. RefAV API의 시간 순서 함수

### 3-a. 공식 프롬프트의 원자 함수 목록
RefProg가 모델에게 보여 주는 API(`run/llm_prompting/RefAV/atomic_functions.txt`)와 코드상의 함수(`refAV/atomic_functions.py`)를 모두 확인합니다. 키워드 검색은 후보를 찾는 보조 수단입니다. **결론은 표의 설명을 직접 읽고 내리십시오.**

In [10]:
REPO = WORK / "RefAV"
if not (REPO / ".git").exists():
    subprocess.run(["git", "clone", "--depth", "1", REFAV_REPO_URL, str(REPO)], check=True)
REFAV_SHA = subprocess.run(["git", "-C", str(REPO), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("RefAV commit:", REFAV_SHA)

prompt_api = REPO / "run/llm_prompting/RefAV/atomic_functions.txt"
if not prompt_api.exists():
    cands = sorted(REPO.rglob("atomic_functions*.txt"))
    assert cands, "프롬프트용 atomic_functions*.txt 를 찾지 못했습니다."
    prompt_api = cands[0]
print("프롬프트 API 파일:", prompt_api.relative_to(REPO))


def list_functions(text):
    # 파이썬으로 파싱되면 ast, 아니면 정규식으로 def와 바로 뒤의 docstring을 추출
    try:
        tree = ast.parse(text)
        out = []
        for n in tree.body:
            if isinstance(n, ast.FunctionDef):
                args = [a.arg for a in n.args.args]
                out.append({"name": n.name, "args": ", ".join(args), "doc": ast.get_docstring(n) or ""})
        if out:
            return out
    except SyntaxError:
        pass
    out = []
    for m in re.finditer(r"^def\s+(\w+)\s*\((.*?)\)\s*(?:->[^:]*)?:\s*\n(\s*(?:\"\"\"|''')(.*?)(?:\"\"\"|'''))?",
                         text, re.S | re.M):
        out.append({"name": m.group(1), "args": " ".join(m.group(2).split()), "doc": (m.group(4) or "").strip()})
    return out


api = pd.DataFrame(list_functions(prompt_api.read_text(encoding="utf-8")))
print("프롬프트 API 함수 수:", len(api))

code_api_path = REPO / "refAV/atomic_functions.py"
code_funcs = set()
if code_api_path.exists():
    code_funcs = {f["name"] for f in list_functions(code_api_path.read_text(encoding="utf-8"))}
    print("코드에만 있고 프롬프트에 없는 함수:", sorted(code_funcs - set(api.name)) or "없음")

# "timestamps", "within" 등은 거의 모든 함수 설명에 나오므로 제외하고, 선후·지속 관계를 나타내는 말만 찾음
TEMPORAL_KW = (r"\b(before|after|prior|preced\w*|follow\w*|subsequent\w*|then|sequen\w*|earlier|later|"
               r"until|since|duration|window|consecutive|history|past|future|min_timesteps?|dilat\w*)\b")
api["first_line"] = api.doc.str.strip().str.split("\n").str[0].str.slice(0, 140)
api["temporal_kw"] = (api.name + " " + api.args + " " + api.doc).str.lower().apply(
    lambda s: ", ".join(sorted(set(re.findall(TEMPORAL_KW, s)))))
pd.set_option("display.max_colwidth", 140)
pd.set_option("display.max_rows", 200)
print("\n[3-a] 전체 함수 (시간 관련 키워드가 걸린 항목은 temporal_kw 열에 표시)")
display(api[["name", "args", "first_line", "temporal_kw"]])

RefAV commit: 5c5be6439ce59b61a31d56431a79a8a04bba33fa
프롬프트 API 파일: run/llm_prompting/RefAV/atomic_functions.txt
프롬프트 API 함수 수: 31
코드에만 있고 프롬프트에 없는 함수: ['within_camera_view']

[3-a] 전체 함수 (시간 관련 키워드가 걸린 항목은 temporal_kw 열에 표시)


,name,args,first_line,temporal_kw
0,has_objects_in_relative_direction,"track_candidates, related_candidates, log_dir, direction, min_number, max_number, within_distance, lateral_thresh",Identifies tracked objects with at least the minimum number of related candidates in the specified direction.,
1,get_objects_in_relative_direction,"track_candidates, related_candidates, log_dir, direction, min_number, max_number, within_distance, lateral_thresh",Returns a scenario dictionary of the related candidates that are in the relative direction of the track candidates.,
2,get_objects_of_category,"log_dir, category",Returns all objects from a given category from the log annotations. This method accepts the,
3,is_category,"track_candidates, log_dir, category",Returns all objects from a given category from track_candidates dict. This method accepts the,
4,is_color,"track_candidates, log_dir, color","Returns objects that are the given color, determined by SIGLIP.",
5,turning,"track_candidates, log_dir, direction",Returns objects that are turning in the given direction.,
6,changing_lanes,"track_candidates, log_dir, direction",Identifies lane change events for tracked objects in a scenario.,
7,has_lateral_acceleration,"track_candidates, log_dir, min_accel, max_accel",Objects with a lateral acceleartion between the minimum and maximum thresholds.,
8,facing_toward,"track_candidates, related_candidates, log_dir, within_angle, max_distance",Identifies objects in track_candidates that are facing toward objects in related candidates.,
9,heading_toward,"track_candidates, related_candidates, log_dir, angle_threshold, minimum_speed, max_distance",Identifies objects in track_candidates that are heading toward objects in related candidates.,


In [11]:
# [3-b] 시간 관련 키워드가 걸린 함수의 설명 전문
hits = api[api.temporal_kw != ""]
for r in hits.itertuples():
    print("=" * 100)
    print(f"{r.name}({r.args})  | 키워드: {r.temporal_kw}")
    print(r.doc)

# [3-c] 프롬프트의 예시 파일에서 before/after/then이 들어간 줄
print("\n" + "=" * 100)
print("[3-c] 프롬프트 디렉터리의 다른 파일에서 before / after / then / until / once 검색")
for f in sorted(prompt_api.parent.rglob("*")):
    if f.is_file() and f != prompt_api and f.suffix in {".txt", ".py", ".md", ".json"}:
        lines = f.read_text(encoding="utf-8", errors="ignore").splitlines()
        found = [(i + 1, l.strip()) for i, l in enumerate(lines)
                 if re.search(r"\b(before|after|then|until|once)\b", l, re.I)]
        if found:
            print(f"--- {f.relative_to(REPO)} ({len(found)}줄)")
            for i, l in found[:15]:
                print(f"  {i:4d}: {l[:160]}")

being_crossed_by(track_candidates, related_candidates, log_dir, direction, in_direction, forward_thresh, lateral_thresh)  | 키워드: past, until
Identifies objects that are being crossed by one of the related candidate objects. A crossing is defined as
the related candidate's centroid crossing the half-midplane of a tracked candidate. The direction of the half-
midplane is specified with the direction. 

Args:
    track_candidates: Tracks to analyze .
    related_candidates: Candidates (e.g., pedestrians or vehicles) to check for crossings.
    log_dir: Path to scenario logs.
    direction: specifies the axis and direction the half midplane extends from 
    in_direction: which direction the related candidate has to cross the midplane for it to be considered a crossing
    forward_thresh: how far the midplane extends from the edge of the tracked object
    lateral_thresh: the two planes offset from the midplane. If an related candidate crosses the midplane, it will 
    continue being cons

### 3-d. RefAV val 설명 중 시간 접속사가 들어간 설명
서론에서 "RefAV 설명 중 순서 관계 질의가 얼마나 되는가"를 말할 근거입니다. `while`은 "차선 변경 중"처럼 상태를 뜻하는 경우가 많으므로 순서 관계(before/after/then 등)와 따로 셉니다.

In [12]:
pairs_path = WORK / "refav_pilot_data" / "log_prompt_pairs_val.json"
if not pairs_path.exists():
    try:
        from huggingface_hub import hf_hub_download
        pairs_path = Path(hf_hub_download(repo_id=HF_REPO, filename="log_prompt_pairs_val.json",
                                          repo_type="dataset", local_dir=str(WORK / "refav_pilot_data")))
    except Exception as exc:
        pairs_path = None
        print(f"HF 다운로드 실패 ({type(exc).__name__}: {exc}). 이 셀은 건너뜁니다.")

if pairs_path:
    log_prompts = json.loads(Path(pairs_path).read_text(encoding="utf-8"))
    descs = sorted({d for v in log_prompts.values() for d in v})
    groups = {
        "순서(before/after/then/followed by/prior to/until/once/subsequently)":
            r"\b(before|after|then|followed by|prior to|until|once|subsequently|afterwards?)\b",
        "동시(while/during/as)": r"\b(while|during|as)\b",
        "변화 동사(start/stop/begin ...ing)": r"\b(starts?|begins?|stops?|resumes?|finishes?)\s+\w+ing\b",
    }
    counts = []
    for label, pat in groups.items():
        m = [d for d in descs if re.search(pat, d, re.I)]
        counts.append({"유형": label, "설명 수": len(m), "비율": round(len(m) / len(descs), 3)})
        print("=" * 100)
        print(f"{label}: {len(m)} / {len(descs)}")
        for d in m[:10]:
            print("  -", d)
    display(pd.DataFrame(counts))
    pd.DataFrame(counts).to_csv(OUT_DIR / "check3_refav_temporal_descriptions.csv", index=False)

log_prompt_pairs_val.json:   0%|          | 0.00/89.2k [00:00<?, ?B/s]

순서(before/after/then/followed by/prior to/until/once/subsequently): 7 / 403
  - ego vehicle being crossed by a jaywalking pedestrian while being followed by another car
  - ego vehicle waiting for solid lane markings to end before making a lane change
  - pedestrian waiting for ego vehicle to pass before crossing the street
  - people exiting a vehicle from both sides at once
  - vehicle being followed by ego vehicle while being overtaken on right
  - vehicle changing lanes but then changing back within a few seconds
  - waiting until it is safe to turn onto the road
동시(while/during/as): 30 / 403
  - bicycle in same lane as a vehicle
  - bicycle sharing lane as a regular vehicle
  - box truck in the same lane as a school bus
  - ego vehicle being crossed by a jaywalking pedestrian while being followed by another car
  - ego vehicle driving during daylight
  - ego vehicle driving during rain
  - ego vehicle driving during snow
  - ego vehicle driving while it is cloudy
  - left lane cha

,유형,설명 수,비율
0,순서(before/after/then/followed by/prior to/until/once/subsequently),7,0.017
1,동시(while/during/as),30,0.074
2,변화 동사(start/stop/begin ...ing),0,0.000


### 3절 결과 읽는 법

- **시간 순서 함수가 없음**: 서론 2문단에 "RefAV의 원자 함수는 같은 시점의 조건을 결합하지만 사건 사이의 선후를 판정하는 함수는 없어, 순서 관계가 들어간 질의에서는 모델이 시각 비교 코드를 직접 작성해야 한다"고 쓸 수 있고, B2가 RefProg 방식의 추상화라는 근거가 됩니다. 인용할 때 커밋 해시를 함께 기록하십시오.
- **시간 순서 함수가 있음**: 그 함수의 인자 순서와 자연어 어순의 관계를 논점으로 바꾸고, "결합되어 있다"는 서술은 고쳐야 합니다.
- **3-d에서 순서 접속사를 쓴 설명이 매우 적음**: RefAV 데이터로는 본 연구의 질문을 검증할 표본이 부족하다는 점이 새 평가 세트를 만드는 이유가 됩니다. 서론 또는 데이터 절에 한 문장으로 적을 수 있습니다.

In [13]:
summary = {"pred_csv": str(csv_path),
           "check1_b2_while": wb.groupby("variant").while_class.value_counts().unstack(fill_value=0).to_dict("index"),
           "check2_v05": summarize5(res5).reset_index().to_dict("records"),
           "check3_refav_commit": REFAV_SHA,
           "check3_temporal_api_candidates": hits.name.tolist()}
(OUT_DIR / "checks_summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2, default=str),
                                              encoding="utf-8")
print(json.dumps(summary, ensure_ascii=False, indent=2, default=str)[:3000])
print("\n저장:", OUT_DIR / "checks_summary.json")

{
  "pred_csv": "/content/drive/MyDrive/AV2_TQP_TEG_Results/v04_val_20261016_045848/llm_predictions_v04.csv",
  "check1_b2_while": {
    "balanced": {
      "FAIL": 0,
      "INTERP": 16,
      "MIXED": 0,
      "NONE": 0,
      "OK": 32
    },
    "counterbalanced": {
      "FAIL": 0,
      "INTERP": 48,
      "MIXED": 0,
      "NONE": 0,
      "OK": 0
    },
    "zero_shot": {
      "FAIL": 10,
      "INTERP": 16,
      "MIXED": 10,
      "NONE": 9,
      "OK": 3
    }
  },
  "check2_v05": [
    {
      "variant": "zero_shot",
      "method": "B2 (Python)",
      "n": 288.0,
      "v04_exact": 0.069,
      "v04_probe_exact": 0.017,
      "v05_interp": 0.073,
      "v05_judge": 0.003,
      "impl_err": 0.069
    },
    {
      "variant": "zero_shot",
      "method": "P (TQP)",
      "n": 288.0,
      "v04_exact": 0.177,
      "v04_probe_exact": 0.177,
      "v05_interp": 0.177,
      "v05_judge": 0.177,
      "impl_err": 0.0
    },
    {
      "variant": "balanced",
      "method": "B

/tmp/ipykernel_613/1873761741.py:9: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return df.groupby(["variant", "method"]).apply(agg).reindex(VARIANT_ORDER, level=0)
